<a href="https://colab.research.google.com/github/KaRyzhova/Code-1/blob/experiment/BFS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Задание 15 — BFS и кратчайший путь."""

from collections import deque
import random


class Graph:
    """Невзвешенный граф на списках смежности (по умолчанию неориентированный)."""

    def __init__(self, directed=False):
        self.directed = directed
        self.adj = {}

    def add_vertex(self, v):
        self.adj.setdefault(v, [])

    def add_edge(self, u, v):
        self.add_vertex(u)
        self.add_vertex(v)
        self.adj[u].append(v)
        if not self.directed:
            self.adj[v].append(u)

    # ---------- обход ----------
    def bfs(self, start):
        """Обход в ширину из start. Возвращает список вершин в порядке посещения
        (только вершины, достижимые из start)."""
        if start not in self.adj:
            raise ValueError(f"Вершина {start!r} отсутствует в графе")
        visited = {start}
        order = []
        queue = deque([start])
        while queue:
            v = queue.popleft()
            order.append(v)
            for w in self.adj[v]:
                if w not in visited:
                    visited.add(w)
                    queue.append(w)
        return order

    def bfs_all(self):
        """Обход несвязного графа: запускаем BFS из каждой непосещённой вершины.
        Возвращает список компонент связности (каждая — список вершин в порядке BFS)."""
        visited = set()
        components = []
        for s in self.adj:
            if s in visited:
                continue
            visited.add(s)
            comp = []
            queue = deque([s])
            while queue:
                v = queue.popleft()
                comp.append(v)
                for w in self.adj[v]:
                    if w not in visited:
                        visited.add(w)
                        queue.append(w)
            components.append(comp)
        return components

    # ---------- кратчайший путь ----------
    def shortest_path(self, start, end):
        """Кратчайший путь start -> end как список вершин.
        Если пути нет — возвращает None."""
        if start not in self.adj or end not in self.adj:
            raise ValueError("Начальная или конечная вершина отсутствует в графе")
        if start == end:
            return [start]

        parent = {start: None}
        queue = deque([start])
        while queue:
            v = queue.popleft()
            for w in self.adj[v]:
                if w not in parent:
                    parent[w] = v
                    if w == end:  # ранний выход
                        return self._restore(parent, end)
                    queue.append(w)
        return None  # end недостижима из start

    @staticmethod
    def _restore(parent, end):
        path = []
        v = end
        while v is not None:
            path.append(v)
            v = parent[v]
        path.reverse()
        return path

    def shortest_distance(self, start, end):
        """Длина кратчайшего пути в рёбрах. Если пути нет — -1."""
        path = self.shortest_path(start, end)
        return -1 if path is None else len(path) - 1


# ======================= ТЕСТЫ (assert) =======================
def make_sample_graph():
    #   1 - 2 - 3        6 - 7      8 (изолированная)
    #   |       |
    #   4 ----- 5
    g = Graph()
    for u, v in [(1, 2), (2, 3), (1, 4), (4, 5), (5, 3), (6, 7)]:
        g.add_edge(u, v)
    g.add_vertex(8)
    return g


def test_bfs_order():
    g = make_sample_graph()
    assert g.bfs(1) == [1, 2, 4, 3, 5]


def test_bfs_only_own_component():
    g = make_sample_graph()
    assert sorted(g.bfs(6)) == [6, 7]
    assert g.bfs(8) == [8]


def test_bfs_all_disconnected():
    g = make_sample_graph()
    comps = g.bfs_all()
    assert len(comps) == 3
    assert sorted(sorted(c) for c in comps) == [[1, 2, 3, 4, 5], [6, 7], [8]]


def test_shortest_path():
    g = make_sample_graph()
    path = g.shortest_path(1, 3)
    assert path[0] == 1 and path[-1] == 3
    assert len(path) == 3  # 1-2-3
    assert g.shortest_path(1, 5) == [1, 4, 5]


def test_shortest_path_same_vertex():
    g = make_sample_graph()
    assert g.shortest_path(2, 2) == [2]


def test_shortest_distance():
    g = make_sample_graph()
    assert g.shortest_distance(1, 5) == 2
    assert g.shortest_distance(1, 3) == 2
    assert g.shortest_distance(1, 1) == 0


def test_no_path():
    g = make_sample_graph()
    assert g.shortest_path(1, 6) is None
    assert g.shortest_distance(1, 7) == -1
    assert g.shortest_path(8, 1) is None
    assert g.shortest_distance(8, 8) == 0


def test_missing_vertex():
    g = make_sample_graph()
    for call in (lambda: g.bfs(100),
                 lambda: g.shortest_path(1, 100),
                 lambda: g.shortest_distance(100, 1)):
        try:
            call()
        except ValueError:
            pass
        else:
            assert False, "ожидался ValueError"


def test_single_vertex_graph():
    g = Graph()
    g.add_vertex("a")
    assert g.bfs("a") == ["a"]
    assert g.shortest_path("a", "a") == ["a"]
    assert g.bfs_all() == [["a"]]


def test_empty_graph():
    g = Graph()
    assert g.bfs_all() == []


def test_directed():
    g = Graph(directed=True)
    g.add_edge(1, 2)
    g.add_edge(2, 3)
    assert g.shortest_distance(1, 3) == 2
    assert g.shortest_distance(3, 1) == -1
    assert g.shortest_path(3, 1) is None


def test_cycle_graph():
    n = 10
    g = Graph()
    for i in range(n):
        g.add_edge(i, (i + 1) % n)
    assert g.shortest_distance(0, 5) == 5
    assert g.shortest_distance(0, 7) == 3  # через 0-9-8-7
    assert len(g.bfs(0)) == n


def test_large_path_graph():
    n = 100_000
    g = Graph()
    for i in range(n - 1):
        g.add_edge(i, i + 1)
    assert g.shortest_distance(0, n - 1) == n - 1
    assert len(g.bfs(0)) == n


def test_random_graph_consistency():
    random.seed(42)
    n, m = 2000, 5000
    g = Graph()
    for v in range(n):
        g.add_vertex(v)
    for _ in range(m):
        g.add_edge(random.randrange(n), random.randrange(n))
    for _ in range(50):
        s, t = random.randrange(n), random.randrange(n)
        path = g.shortest_path(s, t)
        if path is None:
            assert t not in g.bfs(s)
            assert g.shortest_distance(s, t) == -1
        else:
            assert path[0] == s and path[-1] == t
            # каждое звено пути — реальное ребро
            for a, b in zip(path, path[1:]):
                assert b in g.adj[a]
            assert g.shortest_distance(s, t) == len(path) - 1


def run_all_tests():
    tests = [(name, f) for name, f in globals().items()
             if name.startswith("test_") and callable(f)]
    for name, f in tests:
        f()
        print(f"OK   {name}")
    print(f"\nВсе тесты пройдены: {len(tests)}")


if __name__ == "__main__":
    # небольшая демонстрация
    g = Graph()
    for u, v in [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E"), ("X", "Y")]:
        g.add_edge(u, v)
    print("bfs(A):", g.bfs("A"))
    print("shortest_path(A, E):", g.shortest_path("A", "E"))
    print("shortest_distance(A, E):", g.shortest_distance("A", "E"))
    print("shortest_path(A, X):", g.shortest_path("A", "X"), "(пути нет)")
    print("shortest_distance(A, X):", g.shortest_distance("A", "X"))
    print()
    run_all_tests()

bfs(A): ['A', 'B', 'C', 'D', 'E']
shortest_path(A, E): ['A', 'B', 'D', 'E']
shortest_distance(A, E): 3
shortest_path(A, X): None (пути нет)
shortest_distance(A, X): -1

OK   test_bfs_order
OK   test_bfs_only_own_component
OK   test_bfs_all_disconnected
OK   test_shortest_path
OK   test_shortest_path_same_vertex
OK   test_shortest_distance
OK   test_no_path
OK   test_missing_vertex
OK   test_single_vertex_graph
OK   test_empty_graph
OK   test_directed
OK   test_cycle_graph
OK   test_large_path_graph
OK   test_random_graph_consistency

Все тесты пройдены: 14
